# Homework 2 — Data Cleaning & Validation

Author: Jiajing Wang

## 1.1 Load and inspect the data

In [1]:
from pathlib import Path
import sqlite3
import pandas as pd
DB_PATH = Path("data/data.db")
if not DB_PATH.is_file():
    raise FileNotFoundError(f"Database not found: {DB_PATH.resolve()}. Place the supplied database at data/data.db and run from this notebook directory.")
# Read the homework and reference tables from a local database copy.
with sqlite3.connect(f"file:{DB_PATH.resolve()}?mode=ro", uri=True) as conn:
    raw = pd.read_sql_query("SELECT * FROM ohlc_hw", conn)
    reference = pd.read_sql_query("SELECT * FROM ohlc", conn)
raw.head()

,ts,open,high,low,close,adj_close,volume,ticker
0,1975-04-14,5.4300,5.4800,5.3600,5.4400,1.640000,446392,CAT
1,1978-04-05,4.8400,4.8900,4.8100,4.8800,0.656990,355200,CVX
2,2018-05-25,124.5400,124.7600,121.2200,122.1900,92.220000,8936649,CVX
3,1995-12-07,9.9700,9.9700,9.8300,9.9400,5.880000,3954150,HD
4,1981-07-23,0.1038,0.1038,0.1038,0.1038,0.080013,34451232,AAPL


### Data dictionary

- `ts`: Observation date.
- `ticker`: Stock identifier; inspect and normalize labels before grouping.
- `open/high/low/close`: Raw opening, high, low, and closing prices.
- `adj_close`: Vendor-adjusted closing price; raw OHLC bounds do not apply to this field.
- `volume`: Trading volume.

In [2]:
raw.info()
display(raw.describe(include="all"))
print("Date range:", raw["ts"].min(), "to", raw["ts"].max())
print("Raw ticker labels:", sorted(raw["ticker"].dropna().unique()))


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 397711 entries, 0 to 397710
Data columns (total 8 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   ts         397711 non-null  object 
 1   open       397711 non-null  float64
 2   high       397711 non-null  float64
 3   low        397711 non-null  float64
 4   close      397711 non-null  float64
 5   adj_close  377856 non-null  float64
 6   volume     397711 non-null  int64  
 7   ticker     397711 non-null  object 
dtypes: float64(5), int64(1), object(2)
memory usage: 24.3+ MB


,ts,open,high,low,close,adj_close,volume,ticker
count,397711,397711.000000,397711.000000,397711.000000,397711.000000,377856.000000,3.977110e+05,397711
unique,15868,NaN,NaN,NaN,NaN,NaN,NaN,150
top,2020-11-24,NaN,NaN,NaN,NaN,NaN,NaN,PG
freq,40,NaN,NaN,NaN,NaN,NaN,NaN,15450
mean,NaN,46.285479,46.747660,45.816279,46.292903,51.652250,3.097773e+07,NaN
std,NaN,67.132594,67.778210,66.477178,67.141879,194.584045,1.266237e+08,NaN
min,NaN,0.000000,0.005208,0.004801,0.005208,0.002945,0.000000e+00,NaN
25%,NaN,4.209938,4.240000,4.160000,4.200000,1.390000,1.730700e+06,NaN
50%,NaN,19.850000,20.060000,19.630000,19.860000,11.960000,4.648200e+06,NaN
75%,NaN,57.530000,58.130000,56.950000,57.530000,43.770000,1.102380e+07,NaN


Date range: 1962-01-02 to 2024-12-06
Raw ticker labels: ['<a target="_blank"><span style="color: blue">AAPL</span></a>', '<a target="_blank"><span style="color: blue">AMGN</span></a>', '<a target="_blank"><span style="color: blue">AMZN</span></a>', '<a target="_blank"><span style="color: blue">AXP</span></a>', '<a target="_blank"><span style="color: blue">BA</span></a>', '<a target="_blank"><span style="color: blue">CAT</span></a>', '<a target="_blank"><span style="color: blue">CRM</span></a>', '<a target="_blank"><span style="color: blue">CSCO</span></a>', '<a target="_blank"><span style="color: blue">CVX</span></a>', '<a target="_blank"><span style="color: blue">DIS</span></a>', '<a target="_blank"><span style="color: blue">GS</span></a>', '<a target="_blank"><span style="color: blue">HD</span></a>', '<a target="_blank"><span style="color: blue">HON</span></a>', '<a target="_blank"><span style="color: blue">IBM</span></a>', '<a target="_blank"><span style="color: blue">JNJ</span></a>

## 1.2 Detect data quality issues

### 1.2.1 Clean Ticker Name

In [3]:
import html
import re

def normalize_ticker(value):
    """Normalize vendor ticker formatting while preserving missing labels."""
    if pd.isna(value):
        return pd.NA
    text = html.unescape(str(value))
    text = re.sub(r"<[^>]*>", "", text)
    text = text.strip().upper()
    return text if text else pd.NA

In [4]:
def normalize_records(frame):
    out = frame.copy().reset_index(drop=True)
    # Keep the original label so changes can be reviewed.
    out['ticker_original'] = out['ticker']
    out['ticker'] = out['ticker'].map(normalize_ticker)
    # Invalid dates become missing; valid dates use YYYY-MM-DD.
    dates = pd.to_datetime(out['ts'], errors='coerce')
    out['ts'] = dates.dt.strftime('%Y-%m-%d')
    return out

In [5]:
# Normalize each input once; preserve raw for comparison.
working = normalize_records(raw)
normalized_reference = normalize_records(reference)


ticker_audit = (
    working.groupby(["ticker_original", "ticker"], dropna=False)
    .size().rename("row_count").reset_index()
)
print("Original labels:", raw["ticker"].nunique())
print("Normalized tickers:", working["ticker"].nunique())
display(ticker_audit.head(15))

Original labels: 150
Normalized tickers: 30


,ticker_original,ticker,row_count
0,"<a target=""_blank""><span style=""color: blue"">A...",AAPL,307
1,"<a target=""_blank""><span style=""color: blue"">A...",AMGN,294
2,"<a target=""_blank""><span style=""color: blue"">A...",AMZN,185
3,"<a target=""_blank""><span style=""color: blue"">A...",AXP,356
4,"<a target=""_blank""><span style=""color: blue"">B...",BA,462
5,"<a target=""_blank""><span style=""color: blue"">C...",CAT,394
6,"<a target=""_blank""><span style=""color: blue"">C...",CRM,129
7,"<a target=""_blank""><span style=""color: blue"">C...",CSCO,230
8,"<a target=""_blank""><span style=""color: blue"">C...",CVX,400
9,"<a target=""_blank""><span style=""color: blue"">D...",DIS,458


### 1.2.2 Exact duplicates and conflicting keys
Exact duplicate: identical values in all original columns. Keep one copy.

Conflicting key: multiple different records for the same `(ticker, ts)`. Need Investigation.

In [6]:
# Remove redundant exact rows, then flag all conflicting ticker/date keys.
data_columns = list(raw.columns)
exact_duplicate_mask = working.duplicated(subset=data_columns, keep="first")
removed = working.loc[exact_duplicate_mask].copy()
removed["removal_reason"] = "redundant_exact_duplicate"
unique_records = working.loc[~exact_duplicate_mask].sort_values(["ticker", "ts"]).copy()
conflicting_key_mask = unique_records.duplicated(
    subset=["ticker", "ts"], keep=False
)
conflicting_records = unique_records.loc[conflicting_key_mask].copy()

print("Input rows:", len(working))
print("Redundant exact-duplicate rows:", int(exact_duplicate_mask.sum()))
print("Rows after exact deduplication:", len(unique_records))
print("Rows with conflicting ticker/date keys:", len(conflicting_records))
display(conflicting_records.head())

Input rows: 397711
Redundant exact-duplicate rows: 29460
Rows after exact deduplication: 368251
Rows with conflicting ticker/date keys: 0


,ts,open,high,low,close,adj_close,volume,ticker,ticker_original


### 1.2.3 Missing-cell diagnostics

Calculate missing counts and missing percentages

In [7]:
def missing_value_summary(frame):
    counts = frame.isna().sum()
    percentages = counts / len(frame) * 100 if len(frame) else counts * float("nan")
    return pd.DataFrame({
        "missing_count": counts,
        "missing_percent": percentages,
    })

print("Original Data Missing:")
display(missing_value_summary(raw))
print("After Normalization Missing:")
display(missing_value_summary(unique_records[data_columns]))

Original Data Missing:


,missing_count,missing_percent
ts,0,0.000000
open,0,0.000000
high,0,0.000000
low,0,0.000000
close,0,0.000000
adj_close,19855,4.992319
volume,0,0.000000
ticker,0,0.000000


After Normalization Missing:


,missing_count,missing_percent
ts,0,0.000000
open,0,0.000000
high,0,0.000000
low,0,0.000000
close,0,0.000000
adj_close,18374,4.989532
volume,0,0.000000
ticker,0,0.000000


### 1.2.4 Diagnostic methods and repair policy

In [8]:
import numpy as np

PRICE_COLUMNS = ['open', 'high', 'low', 'close', 'adj_close']
RAW_PRICES = PRICE_COLUMNS[:4]
DATA_COLUMNS = ['ts', *PRICE_COLUMNS, 'volume', 'ticker']

### `missing_trading_date`

In [9]:
def missing_sessions(frame, reference):
    # Use supplied trading dates, excluding weekends and exchange holidays.
    expected = reference[['ticker', 'ts']].dropna().drop_duplicates()
    observed = frame[['ticker', 'ts']].dropna().drop_duplicates()
    comparison = expected.merge(observed, on=['ticker', 'ts'], how='left', indicator=True)
    # left_only means expected in the reference but absent from our data.
    gaps = comparison.loc[comparison['_merge'].eq('left_only'), ['ticker', 'ts']]
    summary = expected.groupby('ticker')['ts'].agg(start='min', end='max')
    summary['missing_days'] = gaps.groupby('ticker').size().reindex(summary.index, fill_value=0)
    summary['description'] = [
        f'Ticker {t} is missing {row.missing_days} trading days between {row.start} and {row.end}'
        for t, row in summary.iterrows()]
    return gaps.reset_index(drop=True), summary.reset_index()

### `Outlier screening`

two-tailed 3-sigma method

In [10]:
def simple_return_screen(frame):
    out = frame.sort_values(['ticker', 'ts']).copy()
    # Shift within each stock: never use another stock's closing price.
    previous = out.groupby('ticker')['close'].shift()
    valid = np.isfinite(out['close']) & out['close'].gt(0) & np.isfinite(previous) & previous.gt(0)
    out['simple_return'] = (out['close'] / previous - 1).where(valid)
    grouped = out.groupby('ticker')['simple_return']
    mean = grouped.transform('mean')
    std = grouped.transform('std')
    # Flag unusually large gains or losses; do not delete automatically.
    out['extreme_return_flag'] = (std.gt(0) & std.notna() & (out['simple_return'] - mean).abs().gt(3 * std))
    return out

### `quality_report`

Multiple issues per observation are allowed.
- **High:** Invalid prices(price should >0), missing prices or trading sessions, or adjustment mismatches.
- **Medium:** outliers, missing volume, or invalid volume.
- **Low:** Zero volume.

In [11]:
def quality_report(frame, reference):
    rows = []
    def add(mask, issue, severity, description, source=frame):
        for row in source.loc[mask].itertuples(index=False):
            detail = description(row) if callable(description) else description
            rows.append((row.ticker, issue, row.ts, detail, severity))

    for col in DATA_COLUMNS:
        add(frame[col].isna(), 'missing_' + col,
            'medium' if col == 'volume' else 'high', f'Missing {col} cell')
        if col in PRICE_COLUMNS:
            v = pd.to_numeric(frame[col], errors='coerce')
            add(v.notna() & (~np.isfinite(v) | v.le(0)), 'invalid_' + col,
                'high', f'{col} must be finite and positive')

    low, high = frame['low'], frame['high']
    bounds = (low.gt(high) | frame.open.lt(low) | frame.open.gt(high)
              | frame.close.lt(low) | frame.close.gt(high))
    add(bounds, 'ohlc_bounds', 'high', 'Raw prices violate low <= open/close <= high')
    volume = frame['volume']
    add(volume.notna() & (~np.isfinite(volume) | volume.lt(0)),
        'invalid_volume', 'medium', 'Volume must be finite and nonnegative')
    add(volume.eq(0), 'zero_volume', 'low', 'Zero volume: review for possible stale data')
    conflicts = frame.duplicated(['ticker', 'ts'], keep=False)
    add(conflicts, 'conflicting_key', 'high', 'Multiple records share a ticker/date key after exact deduplication')

    # outliers screening
    if not conflicts.any():
        screened = simple_return_screen(frame)
        add(screened.extreme_return_flag, 'extreme_simple_return', 'medium',
            lambda r: f'Simple raw-close return {r.simple_return:.8f} exceeds the ticker 3-sigma screen; review, not automatic deletion', screened)

    # Compare each observation with the factor for the same ticker/date.
    trusted = reference[['ticker', 'ts', 'close', 'adj_close']].rename(
        columns={'close': 'reference_close', 'adj_close': 'reference_adjusted'})
    check = frame.merge(trusted, on=['ticker', 'ts'], how='left', validate='many_to_one')
    expected = check['close'] * (check.reference_adjusted / check.reference_close)
    mismatch = (np.isfinite(expected) & expected.gt(0) & check.adj_close.notna()
                & ~np.isclose(check.adj_close, expected, rtol=1e-6, atol=1e-8))
    add(mismatch, 'adjustment_mismatch', 'high',
        'Adjusted close differs from raw close times the matched supplied-reference adjustment factor', check)
    gaps, _ = missing_sessions(frame, reference)
    add(gaps.ts.notna(), 'missing_session', 'high',
        'Observation absent from the supplied per-ticker reference calendar', gaps)
    return pd.DataFrame(rows, columns=['ticker', 'issue_type', 'issue_date', 'description', 'severity'])


Final Result

In [12]:
# Diagnose the normalized, deduplicated records already prepared above.
gaps, gap_summary = missing_sessions(unique_records, normalized_reference)
display(gap_summary[["ticker", "missing_days", "description"]])
quality_before = quality_report(unique_records, normalized_reference)
display(quality_before.groupby(["issue_type", "severity"]).size().rename("issue_count").to_frame())
display(quality_before)

,ticker,missing_days,description
0,AAPL,0,Ticker AAPL is missing 0 trading days between ...
1,AMGN,0,Ticker AMGN is missing 0 trading days between ...
2,AMZN,0,Ticker AMZN is missing 0 trading days between ...
3,AXP,0,Ticker AXP is missing 0 trading days between 1...
4,BA,0,Ticker BA is missing 0 trading days between 19...
5,CAT,0,Ticker CAT is missing 0 trading days between 1...
6,CRM,0,Ticker CRM is missing 0 trading days between 2...
7,CSCO,0,Ticker CSCO is missing 0 trading days between ...
8,CVX,0,Ticker CVX is missing 0 trading days between 1...
9,DIS,0,Ticker DIS is missing 0 trading days between 1...


,,issue_count
issue_type,severity,
adjustment_mismatch,high,10417
extreme_simple_return,medium,5056
invalid_open,high,5
missing_adj_close,high,18374
ohlc_bounds,high,5
zero_volume,low,75


,ticker,issue_type,issue_date,description,severity
0,CVX,invalid_open,1962-01-02,open must be finite and positive,high
1,JNJ,invalid_open,1962-01-02,open must be finite and positive,high
2,MCD,invalid_open,1966-07-05,open must be finite and positive,high
3,MRK,invalid_open,1962-01-02,open must be finite and positive,high
4,PG,invalid_open,1962-01-02,open must be finite and positive,high
...,...,...,...,...,...
33927,WMT,adjustment_mismatch,2024-02-12,Adjusted close differs from raw close times th...,high
33928,WMT,adjustment_mismatch,2024-02-15,Adjusted close differs from raw close times th...,high
33929,WMT,adjustment_mismatch,2024-07-26,Adjusted close differs from raw close times th...,high
33930,WMT,adjustment_mismatch,2024-08-14,Adjusted close differs from raw close times th...,high


## 1.3 Clean the Data

### 1.3.1 Continue from Section 1.2

In [13]:
# Continue from Section 1.2; reuse removed records and quality_before.
work = unique_records.copy()
quarantine_parts = []

### 1.3.2 Obtain invalid or conflicting keys

Conflicted ticker/date records found in 1.2 are grouped here.

In [14]:
def quarantine_rows(frame, mask, reason):
    """Return retained rows and rejected rows with a documented reason."""
    rejected = frame.loc[mask].copy()
    rejected["quarantine_reason"] = reason
    return frame.loc[~mask].copy(), rejected

work, rejected = quarantine_rows(work, work[["ticker", "ts"]].isna().any(axis=1), "invalid_or_missing_key")
quarantine_parts.append(rejected)
work, rejected = quarantine_rows(work, work.index.isin(conflicting_records.index), "unresolved_conflicting_key")
quarantine_parts.append(rejected)

### 1.3.3 Drop rows where all raw prices are missing

In [15]:
def drop_all_missing_prices(frame):
    """Separate all-missing raw OHLC rows before filling."""
    mask = frame[RAW_PRICES].isna().all(axis=1)
    dropped = frame.loc[mask].copy()
    dropped["removal_reason"] = "all_raw_prices_missing_before_fill"
    return frame.loc[~mask].copy(), dropped

work, dropped = drop_all_missing_prices(work)
removed = pd.concat([removed, dropped], ignore_index=True)
print("All-missing raw-price rows removed:", len(dropped))

All-missing raw-price rows removed: 0


### 1.3.4 Forward-fill missing prices within each ticker

In [16]:
def fill_missing_prices(frame):
    """Forward-fill raw OHLC within sorted ticker groups and flag affected rows."""
    out = frame.sort_values(["ticker", "ts"]).copy()
    out["raw_price_filled"] = out[RAW_PRICES].isna().any(axis=1)
    out[RAW_PRICES] = out.groupby("ticker")[RAW_PRICES].ffill()
    return out

work = fill_missing_prices(work)
print("Rows flagged for raw-price filling:", int(work.raw_price_filled.sum()))

Rows flagged for raw-price filling: 0


### 1.3.5 Fill missing volume with zero and keep a flag

In [17]:
def fill_missing_volume(frame):
    """Fill unknown volume with zero while preserving an imputation flag."""
    out = frame.copy()
    out["volume_filled"] = out["volume"].isna()
    out["volume"] = out["volume"].fillna(0)
    return out

work = fill_missing_volume(work)
print("Missing volume cells filled:", int(work.volume_filled.sum()))

Missing volume cells filled: 0


### 1.3.6 Check invalid prices and impossible OHLC relationships

All prices must be finite and positive, with `low <= open <= high` and `low <= close <= high`.

In [18]:
def valid_raw_prices(frame):
    """Identify rows with finite positive raw prices and valid OHLC bounds."""
    prices = frame[RAW_PRICES]
    return (np.isfinite(prices).all(axis=1) & prices.gt(0).all(axis=1)
            & frame["low"].le(frame["high"])
            & frame["open"].between(frame["low"], frame["high"])
            & frame["close"].between(frame["low"], frame["high"]))

print("Raw-price rows requiring review:", int((~valid_raw_prices(work)).sum()))

Raw-price rows requiring review: 5


### 1.3.7 Repair wrong open/close values using a previous valid close
Using the previous validated close only when it lies within today's valid price range. No future value will be used to repair. Unresolved high/low errors and leading gaps will be dropped, but save in backup dataset.

In [19]:
def repair_price_bounds(frame):
    """Repair invalid open/close only when a prior valid close fits today's range."""
    out = frame.copy()
    previous = out["close"].where(valid_raw_prices(out))
    previous = previous.groupby(out["ticker"]).ffill()
    previous = previous.groupby(out["ticker"]).shift()
    # A previous close is usable only if it fits today's high/low range.
    usable = previous.gt(0) & previous.between(out["low"], out["high"])
    out["raw_bounds_repaired"] = False
    for col in ["open", "close"]:
        bad = ~np.isfinite(out[col]) | out[col].le(0) | ~out[col].between(out["low"], out["high"])
        repair = bad & usable
        out.loc[repair, col] = previous.loc[repair]
        out.loc[repair, "raw_bounds_repaired"] = True
    return out

work = repair_price_bounds(work)

### 1.3.8 Quarantine invalid prices and volume

In [20]:
work, rejected = quarantine_rows(work, ~valid_raw_prices(work), "unresolved_raw_price_or_bounds; no_defensible_previous_close")
quarantine_parts.append(rejected)
invalid_volume = ~np.isfinite(work["volume"]) | work["volume"].lt(0)
work, rejected = quarantine_rows(work, invalid_volume, "invalid_volume")
quarantine_parts.append(rejected)
print("Rows repaired using a previous valid close:", int(work.raw_bounds_repaired.sum()))

Rows repaired using a previous valid close: 0


### 1.3.9 Repair adjusted prices

In [21]:
# Match the date-specific reference factor.
trusted = normalized_reference[["ticker", "ts", "close", "adj_close"]].rename(
    columns={"close": "reference_close", "adj_close": "reference_adjusted"})
work = work.merge(trusted, on=["ticker", "ts"], how="left", validate="one_to_one")
work["adjustment_factor"] = work["reference_adjusted"] / work["reference_close"]
trusted_factor = (np.isfinite(work["adjustment_factor"]) & work["adjustment_factor"].gt(0)
                  & np.isfinite(work["reference_close"]) & work["reference_close"].gt(0))

In [22]:
def repair_adjusted_prices(frame, trusted_factor):
    """Reconstruct adjusted close only with a trusted factor for the same date."""
    out = frame.copy()
    # Adjusted close = repaired raw close × the same-date reference factor.
    expected = out["close"] * out["adjustment_factor"]
    out["adj_close_original"] = out["adj_close"]
    mismatch = ~np.isclose(out["adj_close"], expected, rtol=1e-6, atol=1e-8)
    out["adjusted_price_repaired"] = (~np.isfinite(out["adj_close"]) | out["adj_close"].le(0)
        | (trusted_factor & mismatch) | out["raw_price_filled"] | out["raw_bounds_repaired"]
    )
    repair = out["adjusted_price_repaired"] & trusted_factor
    out.loc[repair, "adj_close"] = expected.loc[repair]
    out["adjustment_source"] = np.where(trusted_factor, "supplied_ohlc_matched_ticker_date", "unverified")
    return out

work = repair_adjusted_prices(work, trusted_factor)
print("Adjusted prices reconstructed:", int(work.adjusted_price_repaired.sum()))

Adjusted prices reconstructed: 28791


In [23]:
unresolved = work["adjusted_price_repaired"] & ~trusted_factor
work, rejected = quarantine_rows(work, unresolved, "unresolved_adjustment_factor")
quarantine_parts.append(rejected)
invalid_adjusted = ~np.isfinite(work["adj_close"]) | work["adj_close"].le(0)
work, rejected = quarantine_rows(work, invalid_adjusted, "invalid_adjusted_price")
quarantine_parts.append(rejected)

### 1.3.10 Review extreme returns

In [24]:
# Keep statistical extremes as review flags unless an error is established.
work = simple_return_screen(work).reset_index(drop=True)
work["extreme_return_review"] = np.where(work["extreme_return_flag"],
    "Retained: statistical flag alone does not establish an error; no corporate-event claim made", "")
print("Retained extreme-return review flags:", int(work.extreme_return_flag.sum()))
display(work.loc[work.extreme_return_flag, ["ticker", "ts", "simple_return"]].head())

Retained extreme-return review flags: 5057


,ticker,ts,simple_return
9,AAPL,1980-12-26,0.092288
76,AAPL,1981-04-02,0.087659
197,AAPL,1981-09-25,-0.129776
318,AAPL,1982-03-19,0.090173
490,AAPL,1982-11-22,-0.089089


### 1.3.11 Check the cleaned data and compare statistics

Run `check_cleaned_data()` on the actual retained dataset. All price, key, volume and OHLC checks must pass before saving. ISO dates and reference adjustment consistency are also checked.

In [25]:
def check_cleaned_data(frame):
    """Check the retained dataset against the assignment's validation requirements."""
    prices = frame[PRICE_COLUMNS]
    volume = frame["volume"]
    matched = frame["adjustment_source"].eq("supplied_ohlc_matched_ticker_date")

    # Check every retained observation, including raw and adjusted prices.
    checks = {
        "Nonempty data": len(frame) > 0,
        "No missing prices": not prices.isna().any().any(),
        "Finite prices": np.isfinite(prices.to_numpy()).all(),
        "Positive prices": prices.gt(0).all().all(),
        "No missing ticker/date keys": not frame[["ticker", "ts"]].isna().any().any(),
        "Unique ticker/date keys": not frame.duplicated(["ticker", "ts"]).any(),
        "Finite volume": np.isfinite(volume).all(),
        "Nonnegative volume": volume.ge(0).all(),
        "Valid OHLC relationships": (frame["low"].le(frame["high"])
            & frame["open"].between(frame["low"], frame["high"])
            & frame["close"].between(frame["low"], frame["high"])).all(),
        "ISO dates": frame["ts"].str.fullmatch(r"\d{4}-\d{2}-\d{2}").fillna(False).all(),
        "Consistent adjusted prices": np.allclose(frame.loc[matched, "adj_close"],
            frame.loc[matched, "close"] * frame.loc[matched, "adjustment_factor"], rtol=1e-6, atol=1e-8),
    }
    # Stop before saving if any validation fails.
    report = pd.Series(checks, name="passed").rename_axis("check").to_frame()
    assert report["passed"].all(), f"Validation failed: {report.index[~report.passed].tolist()}"
    return report


In [26]:
cleaned = work.drop(columns=["reference_close", "reference_adjusted"]).copy()
display(check_cleaned_data(cleaned))

,passed
check,
Nonempty data,True
No missing prices,True
Finite prices,True
Positive prices,True
No missing ticker/date keys,True
Unique ticker/date keys,True
Finite volume,True
Nonnegative volume,True
Valid OHLC relationships,True


**Row accounting:** reconcile all input observations and retain quarantine reasons.

In [27]:
# Reconcile all input rows and display quarantine reasons.
quarantine = pd.concat(quarantine_parts, ignore_index=True)
accounting = pd.DataFrame({"category": ["input", "retained", "removed", "quarantined"],
    "rows": [len(raw), len(cleaned), len(removed), len(quarantine)]})
assert len(raw) == len(cleaned) + len(removed) + len(quarantine)
display(accounting)
display(quarantine[["ticker", "ts", "quarantine_reason"]])

,category,rows
0,input,397711
1,retained,368246
2,removed,29460
3,quarantined,5


,ticker,ts,quarantine_reason
0,CVX,1962-01-02,unresolved_raw_price_or_bounds; no_defensible_...
1,JNJ,1962-01-02,unresolved_raw_price_or_bounds; no_defensible_...
2,MCD,1966-07-05,unresolved_raw_price_or_bounds; no_defensible_...
3,MRK,1962-01-02,unresolved_raw_price_or_bounds; no_defensible_...
4,PG,1962-01-02,unresolved_raw_price_or_bounds; no_defensible_...


**Before/after statistics:** compare unique records before repair with the retained cleaned data, then rerun the issue report.

In [28]:
# Compare before/after statistics and rerun diagnosis.
columns = PRICE_COLUMNS + ["volume"]
display(pd.concat({"before": unique_records[columns].describe(),
                   "after": cleaned[columns].describe()}, axis=1))
quality_after = quality_report(cleaned, normalized_reference)
display(quality_after.groupby(["issue_type", "severity"]).size().rename("issue_count").to_frame())

before                                               \
                open           high            low          close   
count  368251.000000  368251.000000  368251.000000  368251.000000   
mean       46.308766      46.771204      45.839768      46.316711   
std        67.143015      67.789068      66.489203      67.154690   
min         0.000000       0.005208       0.004801       0.005208   
25%         4.210000       4.250000       4.170000       4.210000   
50%        19.890000      20.090000      19.670000      19.900000   
75%        57.560000      58.160000      56.990000      57.570000   
max       604.260000     608.630000     597.880000     605.400000   

                                            after                 \
           adj_close        volume           open           high   
count  349877.000000  3.682510e+05  368246.000000  368246.000000   
mean       51.711536  3.095142e+07      46.309394      46.771824   
std       195.001654  1.266145e+08      67.143254      67.789319   
min         0.002945  0.000000e+00       0.005208       0.005208   
25%         1.395997  1.731400e+06       4.210000       4.250000   
50%        11.980000  4.649600e+06      19.890000      20.090000   
75%        43.790000  1.102151e+07      57.560000      58.160000   
max     10442.510783  9.276606e+09     604.260000     608.630000   

                                                                  
                 low          close      adj_close        volume  
count  368246.000000  368246.000000  368246.000000  3.682460e+05  
mean       45.840375      46.317325      37.274670  3.095184e+07  
std        66.489450      67.154939      63.587872  1.266153e+08  
min         0.004801       0.005208       0.002945  0.000000e+00  
25%         4.170000       4.210000       1.310000  1.731514e+06  
50%        19.670000      19.910000      11.460000  4.649648e+06  
75%        56.997500      57.570000      41.570000  1.102160e+07  
max       597.880000     605.400000     605.400000  9.276606e+09

,,issue_count
issue_type,severity,
extreme_simple_return,medium,5057
missing_session,high,5
zero_volume,low,74


### 1.3.12 Save the cleaned data

Save `ohlc_hw_cleaned` with repair flags in the local database. Keep the raw tables unchanged. 

In [29]:
# Save only the required cleaned table; keep the source table unchanged.
with sqlite3.connect(DB_PATH) as conn:
    raw_count = conn.execute("SELECT COUNT(*) FROM ohlc_hw").fetchone()[0]
    cleaned.to_sql("ohlc_hw_cleaned", conn, if_exists="replace", index=False)
    conn.execute("CREATE UNIQUE INDEX IF NOT EXISTS cleaned_ticker_ts ON ohlc_hw_cleaned(ticker, ts)")
    assert conn.execute("SELECT COUNT(*) FROM ohlc_hw").fetchone()[0] == raw_count
    saved = pd.read_sql_query("SELECT * FROM ohlc_hw_cleaned ORDER BY ticker, ts", conn)
print("Saved cleaned rows:", len(saved))

Saved cleaned rows: 368246


In [30]:
# Verify that database read-back matches the validated result.
pd.testing.assert_frame_equal(saved, cleaned.reset_index(drop=True), check_dtype=False)
print("Passed: the saved table matches the validated cleaned data.")

Passed: the saved table matches the validated cleaned data.


### 1.3.13 Verify required cases with small examples

In [31]:
example = pd.DataFrame({"ticker": ["TEST"] * 4,
    "ts": ["2024-01-02", "2024-01-03", "2024-01-04", "2024-01-05"],
    "open": [100., 101., 102., 103.], "high": [110.] * 4, "low": [90.] * 4,
    "close": [100., 101., 102., 103.], "adj_close": [80., 80.8, 81.6, 82.4],
    "volume": [100.] * 4})
gaps, _ = missing_sessions(example.drop(index=1), example)
assert gaps["ts"].tolist() == ["2024-01-03"]
print("Passed: one deliberately removed trading date was detected.")

Passed: one deliberately removed trading date was detected.


#### Example A: Drop all-missing raw prices before forward fill

In [32]:
test = example.copy()
test.loc[1, RAW_PRICES] = np.nan
kept, dropped = drop_all_missing_prices(test)
assert len(dropped) == 1 and len(kept) == 3
print("Passed: an all-missing OHLC row is removed before filling.")

Passed: an all-missing OHLC row is removed before filling.


#### Example B: Grouped fill, missing volume and leading gaps

In [33]:
test = example.copy()
test.loc[0, "open"] = np.nan
test.loc[2, "open"] = np.nan
test.loc[2, "volume"] = np.nan
filled = fill_missing_volume(fill_missing_prices(test))
assert pd.isna(filled.loc[0, "open"])
assert filled.loc[2, "open"] == 101. and filled.loc[2, "raw_price_filled"]
assert filled.loc[2, "volume"] == 0 and filled.loc[2, "volume_filled"]
print("Passed: valid grouped fill, flagged volume fill and unresolved leading gap.")

Passed: valid grouped fill, flagged volume fill and unresolved leading gap.


#### Example C: No fill across ticker boundaries

In [34]:
other = example.iloc[:1].copy()
other["ticker"] = "ZZZ"
other["open"] = np.nan
filled = fill_missing_prices(pd.concat([example, other], ignore_index=True))
assert filled.loc[filled["ticker"].eq("ZZZ"), "open"].isna().all()
print("Passed: a new ticker cannot inherit another ticker's price.")

Passed: a new ticker cannot inherit another ticker's price.


#### Example D: Repair an impossible close and quarantine an invalid range

In [35]:
test = example.copy()
test.loc[1, "close"] = 999.
test.loc[2, "high"] = 80.
repaired = repair_price_bounds(test)
assert repaired.loc[1, "close"] == 100. and repaired.loc[1, "raw_bounds_repaired"]
kept, rejected = quarantine_rows(repaired, ~valid_raw_prices(repaired), "invalid_raw_bounds")
assert rejected["ts"].tolist() == ["2024-01-04"]
print("Passed: defensible close repair and unresolved-range quarantine.")

Passed: defensible close repair and unresolved-range quarantine.


#### Example E: Adjusted-price reconstruction requires a trusted factor

In [36]:
test = fill_missing_volume(fill_missing_prices(example))
test["raw_bounds_repaired"] = False
test["adjustment_factor"] = [0.8, 0.8, np.nan, 0.8]
test.loc[[1, 2], "adj_close"] = np.nan
trusted_test = test["adjustment_factor"].notna()
repaired = repair_adjusted_prices(test, trusted_test)
assert np.isclose(repaired.loc[1, "adj_close"], 80.8)
assert repaired.loc[1, "adjusted_price_repaired"]
unresolved = repaired["adjusted_price_repaired"] & ~trusted_test
assert unresolved.tolist() == [False, False, True, False]
print("Passed: matched-factor repair and quarantine flag without a trusted factor.")

Passed: matched-factor repair and quarantine flag without a trusted factor.


#### Example F: Conflicting keys and invalid volume are quarantined

In [37]:
test = pd.concat([example, example.iloc[[1]]], ignore_index=True)
test.loc[4, "close"] = 105.
kept, rejected = quarantine_rows(test, test.duplicated(["ticker", "ts"], keep=False), "conflicting_key")
assert len(rejected) == 2
test = example.copy()
test.loc[1, "volume"] = -1
kept, rejected = quarantine_rows(test, ~np.isfinite(test["volume"]) | test["volume"].lt(0), "invalid_volume")
assert len(rejected) == 1
print("Passed: all conflicting records and invalid volume are isolated.")

Passed: all conflicting records and invalid volume are isolated.


### 1.3 Output

- `data/data.db` contains the required `ohlc_hw_cleaned` table . The original source tables remain unchanged. 

- 01_data_cleaning.ipynb with your analysis and code